In [0]:
# Databricks notebook source
# ============================================================
# WORKFLOW 2 — GOLD LAYER
# Reads dependent_jobs from each gold job JSON to build
# execution waves dynamically — no hardcoded dependencies.
# ETL_NB path also read from job config.
# ============================================================


In [0]:
# MAGIC %md
# MAGIC ## 🥇 Workflow 2 — Gold Layer
# MAGIC 
# MAGIC Wave execution is **derived at runtime** from `dependent_jobs` in each gold JSON:
# MAGIC 
# MAGIC - **Wave 1**: Gold jobs with no gold-level dependencies (only silver deps)
# MAGIC - **Wave 2**: Gold jobs that depend on Wave 1 gold jobs
# MAGIC - **Wave 3+**: Further dependency levels if they exist
# MAGIC 
# MAGIC The `ETL_NB` path for each job is read from the JSON — no hardcoded paths.


In [0]:
# COMMAND ----------
import json, os
from concurrent.futures import ThreadPoolExecutor, as_completed
from pyspark.sql.functions import date_format, current_timestamp

dbutils.widgets.text("partition", "20260317173222272")
partition = dbutils.widgets.get("partition")
if not partition:
    partition = str(spark.range(1).select(
        date_format(current_timestamp(), "yyyyMMddHHmmssSSS")
        .cast("bigint").alias("p")
    ).collect()[0]["p"])

BASE_PATH = "/Workspace/Users/sahil.prusty09@gmail.com/ecommerce_customer_intelligence_platform"
JOBS_PATH = f"{BASE_PATH}/jobs/gold"

print(f"{'='*60}")
print(f"  WORKFLOW 2 — GOLD LAYER")
print(f"  Partition  : {partition}")
print(f"  Jobs path  : {JOBS_PATH}")
print(f"{'='*60}")


In [0]:
# COMMAND ----------
# MAGIC %md ## Build Dependency Graph from Job Configs
# MAGIC
# MAGIC Reads `dependent_jobs` from each gold JSON.
# MAGIC Gold-level dependencies determine wave assignment.
# MAGIC Silver dependencies are informational only — already complete after Workflow 1.


In [0]:
# COMMAND ----------
# ── Load all gold job configs ─────────────────────────────────────────
gold_jobs = {}  # job_id → job_config

for fname in sorted(os.listdir(JOBS_PATH)):
    if not fname.endswith(".json"):
        continue
    with open(f"{JOBS_PATH}/{fname}") as f:
        job = json.load(f)
    job["partition"] = partition
    gold_jobs[job["job_id"]] = job

print(f"  Loaded {len(gold_jobs)} gold job config(s):\n")
print(f"  {'job_id':<8} {'job_name':<40} {'ETL_NB':<30} gold_deps")
print("  " + "─"*100)

for jid, jcfg in sorted(gold_jobs.items()):
    gold_deps = [
        str(d["job_id"]) for d in jcfg.get("dependent_jobs", [])
        if d.get("job_type") == "gold"
    ]
    print(f"  {jid:<8} {jcfg['job_name']:<40} {jcfg['ETL_NB']:<30} {', '.join(gold_deps) or 'none'}")


In [0]:
# COMMAND ----------
# MAGIC %md ## Compute Execution Waves
# MAGIC
# MAGIC **Wave assignment algorithm:**
# MAGIC 1. A job with no gold `dependent_jobs` → Wave 1
# MAGIC 2. A job depending on Wave 1 jobs → Wave 2
# MAGIC 3. Iterates until all jobs are assigned


In [0]:
# COMMAND ----------
all_gold_ids = set(gold_jobs.keys())

# Build gold-only dependency map
# (silver deps are already complete — only gold deps affect wave order)
gold_deps_map = {}
for jid, jcfg in gold_jobs.items():
    gold_deps_map[jid] = {
        d["job_id"]
        for d in jcfg.get("dependent_jobs", [])
        if d.get("job_type") == "gold" and d["job_id"] in all_gold_ids
    }

# ── Topological wave assignment ───────────────────────────────────────
waves        = []
assigned     = set()
remaining    = set(all_gold_ids)

while remaining:
    # Jobs whose gold deps are all already assigned
    wave = {
        jid for jid in remaining
        if gold_deps_map[jid].issubset(assigned)
    }
    if not wave:
        raise Exception(
            f"Circular dependency detected in gold jobs.\n"
            f"Remaining unresolved: {remaining}\n"
            f"Deps map: {gold_deps_map}"
        )
    waves.append(sorted(wave))
    assigned  |= wave
    remaining -= wave

print(f"  Computed {len(waves)} execution wave(s):\n")
for i, wave in enumerate(waves, 1):
    names = [gold_jobs[j]['job_name'] for j in wave]
    print(f"  Wave {i}: {wave}")
    for name in names:
        print(f"         → {name}")
    print()


In [0]:
# COMMAND ----------
# MAGIC %md ## Execute Waves Sequentially, Jobs Within Wave in Parallel


In [0]:
# COMMAND ----------
def run_gold_job(job_config):
    job_id   = job_config["job_id"]
    job_name = job_config["job_name"]
    # ── ETL_NB path from job config — not hardcoded ───────────────────
    etl_nb   = f"{BASE_PATH}/{job_config['ETL_NB']}"
    try:
        print(f"  [{job_id}] 🥇 Starting Gold — {job_name}")
        print(f"  [{job_id}]    ETL NB: {job_config['ETL_NB']}")
        dbutils.notebook.run(
            etl_nb, 500,
            {"job_parameters": json.dumps(job_config)}
        )
        print(f"  [{job_id}] ✅ Gold complete — {job_name}")
        return job_id, job_name, "SUCCESS"
    except Exception as e:
        err = f"FAILED: {str(e)}"
        print(f"  [{job_id}] ❌ {job_name}: {err}")
        return job_id, job_name, err

def run_wave(wave_num, job_ids):
    print(f"\n  ── Wave {wave_num} ({len(job_ids)} job(s)) ──")
    results = []
    for job_id in job_ids:
        job = gold_jobs[job_id]
        run_gold_job(job)
    results.append((job['job_id'], job['job_name'], "SUCCESS"))
    failed = [(j, n, s) for j, n, s in results if "FAILED" in s]
    if failed:
        raise Exception(
            f"Wave {wave_num} failed — cannot proceed.\n"
            f"Failed jobs: {[(j, n) for j, n, _ in failed]}"
        )
    print(f"  ✅ Wave {wave_num} complete")
    return results


In [0]:
# COMMAND ----------
all_results = []
for i, wave_jobs in enumerate(waves, 1):
    all_results += run_wave(i, wave_jobs)


In [0]:
# COMMAND ----------
print(f"\n{'='*60}")
print(f"  WORKFLOW 2 SUMMARY")
print(f"{'='*60}")
print(f"  Total waves : {len(waves)}")
print(f"  Total jobs  : {len(all_results)}")
print(f"  Succeeded   : {sum(1 for _,_,s in all_results if s=='SUCCESS')}")
print()
for jid, name, status in sorted(all_results, key=lambda x: x[0]):
    icon = "✅" if status == "SUCCESS" else "❌"
    print(f"  {icon} [{jid}] {name:<45} {status}")

print(f"\n  ✅ Workflow 2 complete — Gold layer ready")
dbutils.notebook.exit(json.dumps({"status": "SUCCESS", "partition": partition}))
